In [288]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

import os
os.chdir("/Users/arttusokka/Documents/AThesis/Prediciting-ATP-matches-outcome/data")

df_raw= pd.read_csv("/Users/arttusokka/Documents/AThesis/Prediciting-ATP-matches-outcome/data/clean_and_raw/atp_matches_2000_2019_raw.csv")

In [289]:
#koko datasetissä 61 732 havaintoa.
len(df_clean)

53100

In [290]:
df_clean["tourney_level"].value_counts()

tourney_level
A    31744
M    10937
G    10099
F      320
Name: count, dtype: int64

In [291]:
df_clean = df_raw.copy()
#Kopioidaan data puhdistusta varten.

#2. Carpet ottelut poistetaan 
df_clean = df_clean[df_clean["surface"].isin(["Hard", "Clay", "Grass"])]
df_clean = df_clean.reset_index(drop=True)
df_clean["surface"].value_counts()
#--------------------------------
#3. in game statistics removed 
#drop the minute column
df_clean = df_clean.drop(columns=["minutes"])

#3. Datasetti sisältää Davis-cup havaintoja. Tutkimuksen tarkoitus on ennustaa ATP-otteluiden tuloksia, joten Davis-cup ottelut poistetaan.
#Lisäksi ingame statseja w_/l_ alkuisia puuttuu lähes yksinomaan vain Davis-cup otteluist.
df_clean = df_clean[df_clean["tourney_level"] != "D"].copy()


#Poistetaan puuttuvat in game statit (w_/l_ alkuisia), näitä on alle 5% datasta.
wl_cols = [col for col in df_clean.columns if col.startswith(("w_", "l_"))]
df_clean = df_clean.dropna(subset=wl_cols).copy()

#3. lisätään vuosi-muuttuja.
df_clean["tourney_date"] = pd.to_datetime(
    df_clean["tourney_date"],
    format="%Y%m%d")

df_clean["year"] = df_clean["tourney_date"].dt.year
df_clean["year"].value_counts().sort_index()

df_clean["tourney_date"].dtype


#
#korjataan virheellinen pituusarvot (3.0cm)
heights = pd.concat([
    df_clean["winner_ht"],
    df_clean["loser_ht"]])

heights.min(), heights.max()

df_clean = df_clean[
    (df_clean["winner_ht"].between(140, 215)) &
    (df_clean["loser_ht"].between(140, 215))]
#-------------------------------
#8. Seed- ja entry-muuttujat sisältävät runsaasti puuttuvia arvoja ja ovat osittain päällekkäisiä ranking-muuttujien kanssa, minkä vuoksi ne poistetaan.
df_clean[[
    "winner_seed", "winner_entry",
    "loser_seed", "loser_entry"
]].isna().mean() * 100

df_clean = df_clean.drop(columns=[
    "winner_seed", "winner_entry",
    "loser_seed", "loser_entry"])

#Kätisyyskorjaukset
#katsotaan A:n (ambidextrous=molempikätisyys) määrät
pd.DataFrame({
    "winner": df_clean["winner_hand"].value_counts(),
    "loser":  df_clean["loser_hand"].value_counts()}).loc[["A", "U"]]
#A yksi havainto (loser), U:ssa 2 havaintoa (winner) ja 7 havaintoa (loser).

pd.DataFrame({
    "winner": df_clean["winner_hand"].value_counts(),
    "loser":  df_clean["loser_hand"].value_counts()}
).loc[["A", "U"]]

#Luke Jenssen on ainoa A-kätinen pelaaja datasetissä.
#katsotaan seuraavaksi Unknown merkatut pelaajat.

u_rows = df_clean[
    (df_clean["winner_hand"] == "U") |
    (df_clean["loser_hand"] == "U")
]


u_players = pd.concat([
    u_rows.loc[u_rows["winner_hand"] == "U", "winner_name"],
    u_rows.loc[u_rows["loser_hand"] == "U", "loser_name"]
]).unique()

u_players
#Guillermo Olaso: right-handed, Christopher Koderisch: ei mainita internetissä jätetään unknown, Jose Hernandez right-handed.

df_clean.loc[df_clean["winner_name"] == "Guillermo Olaso", "winner_hand"] = "R"
df_clean.loc[df_clean["loser_name"] == "Guillermo Olaso", "loser_hand"] = "R"

df_clean.loc[df_clean["winner_name"] == "Jose Hernandez", "winner_hand"] = "R"
df_clean.loc[df_clean["loser_name"] == "Jose Hernandez", "loser_hand"] = "R"

#rankingit
#13% winner_rank (ja winner_rank_points) puuttuu 41% ja loser_rank (ja loser_rank_points) puuttuu 13%
df_clean[[
    "winner_rank", "loser_rank",
    "winner_rank_points", "loser_rank_points"
]].isna().mean() * 100


winner_rank           0.067797
loser_rank            0.227872
winner_rank_points    0.067797
loser_rank_points     0.227872
dtype: float64

In [292]:
df_clean.isna().sum()[df_clean.isna().sum() > 0]

winner_rank            36
winner_rank_points     36
loser_rank            121
loser_rank_points     121
dtype: int64

In [293]:
rank_by_player = pd.concat([
    df_clean[["winner_name", "winner_rank"]]
        .rename(columns={"winner_name": "player", "winner_rank": "rank"}),
    df_clean[["loser_name", "loser_rank"]]
        .rename(columns={"loser_name": "player", "loser_rank": "rank"})])

no_rank_players = (
    rank_by_player
    .groupby("player")["rank"]
    .apply(lambda x: x.notna().any())
    .loc[lambda x: x == False])

no_rank_players.index.tolist()
pd.DataFrame({"player": no_rank_players.index})
#41 pelaajaa, joilla ei ole ranking-historiaa lainkaan.


pd.crosstab(
    df_clean["winner_rank"].isna(),
    df_clean["winner_rank"].shift(1).isna())
#56 k havaintoa ranking ok. 74:ssä ranking ajanhetkellä t ok, t-1 puuttuu. 73 nyt t puuttuu, t-1 taas ok. 2 puuttuu molemmissa (t, t-1).

df_clean[
    df_clean["winner_name"].isin(no_rank_players.index) |
    df_clean["loser_name"].isin(no_rank_players.index)
][["tourney_date", "winner_name", "loser_name", "winner_rank", "loser_rank"]]


,tourney_date,winner_name,loser_name,winner_rank,loser_rank
75,2000-05-01,Martin Damm Sr,Thomas Johansson,NaN,58.0
84,2000-05-01,Franco Squillari,Martin Damm Sr,52.0,NaN
537,2000-01-10,Karol Kucera,Martin Damm Sr,17.0,NaN
652,2000-03-06,Gaston Etlis,Pablo Gonzalez,137.0,NaN
713,2000-02-14,Cecil Mamiit,Luke Jensen,135.0,NaN
756,2000-03-20,Wayne Ferreira,Martin Damm Sr,44.0,NaN
1195,2000-08-14,David Prinosil,Martin Damm Sr,64.0,NaN
1302,2000-07-31,Harel Levy,Martin Damm Sr,144.0,NaN
1373,2000-08-07,Martin Damm Sr,Vincent Spadea,NaN,83.0
1398,2000-08-07,Arnaud Clement,Martin Damm Sr,50.0,NaN


In [294]:

#imputoidaan puuttuvat rankingit pelaajan edellisellä tunnetulla rankingillä.
df_clean = df_clean.sort_values("tourney_date")

df_clean["winner_rank"] = (
    df_clean
    .groupby("winner_name")["winner_rank"]
    .ffill())

df_clean["loser_rank"] = (
    df_clean
    .groupby("loser_name")["loser_rank"]
    .ffill())

#aikaisemmat 41 pelaajaa jakautuvat 33 (winner) ja 115 (loser) otteluhavaintoon. Jätetään ne vielä toistaiseksi sellaisenaan, kuten ylhäällä todettiin.
print(df_clean[["winner_rank", "loser_rank"]].isna().sum())

#tehdään sama rank_points-sarakkeille.
df_clean["winner_rank_points"] = (
    df_clean
    .groupby("winner_name")["winner_rank_points"]
    .ffill())

df_clean["loser_rank_points"] = (
    df_clean
    .groupby("loser_name")["loser_rank_points"]
    .ffill())

#nyt tulee enää pelaajat, joilla ole lainkaan ranking-historiaa. Jätetään nämä sellaisenaan.
df_clean[[
    "winner_rank", "loser_rank",
    "winner_rank_points", "loser_rank_points"
]].isna().sum()

winner_rank    17
loser_rank     64
dtype: int64


winner_rank           17
loser_rank            64
winner_rank_points    17
loser_rank_points     64
dtype: int64

In [295]:
#Jos jollain pelaajalla ei ole ranking:ia yhdessäkään pelissä, niin tällöin voi olettaa, että pelejä ei ole kovinkaan montaa tai pelaajan ranking taso on pieni.
#ranking-arvo = pelaajan sijoitus ATP listalla -> 2000. (teoriassa sama kuin viimeinen)
#ranking-pisteet = pelaajan ATP pisteet -> 0

#alin mahdollinen ranking
df_clean[["winner_rank", "loser_rank"]] = (
    df_clean[["winner_rank", "loser_rank"]].fillna(2000))

# alin mahdollinen ranking-pistemäärä
df_clean[["winner_rank_points", "loser_rank_points"]] = (
    df_clean[["winner_rank_points", "loser_rank_points"]].fillna(0))

df_clean[[
    "winner_rank","loser_rank",
    "winner_rank_points","loser_rank_points"]].isna().sum()

winner_rank           0
loser_rank            0
winner_rank_points    0
loser_rank_points     0
dtype: int64

In [296]:
#print missing values if there is > 0 missing.
df_clean.isna().sum()[df_clean.isna().sum() > 0]

Series([], dtype: int64)

In [297]:
#save the data
df_clean.to_csv("/Users/arttusokka/Documents/AThesis/Prediciting-ATP-matches-outcome/data/clean_and_raw/atp_matches_2000_2019_clean.csv", index=False)
